# Scheduler 6 Experiment

## Pre-run Predictions

These predictions were recorded before running the build or any workload. Scheduler 6 is expected to behave like round robin with a 5-tick quantum. Turnaround means mean completion minus arrival for batch jobs; response means mean ready-to-dispatch delay across all CPU bursts; fairness is the nice-weighted Jain score for batch jobs during their overlap window.

| Workload | Turnaround: best / worst | Response: best / worst | Fairness prediction | Rationale |
|---|---|---|---|---|
| 1: equal-nice batch jobs | 0 and 2 tie / 1 | 1 / 0 and 2 tie | Similar across all four | ID 2's equal-nice tie-breaking behaves like RR 0. A short quantum should improve arrival response; longer quantum limits switch overhead. |
| 2: batch and I/O jobs, equal nice | 0 and 2 tie / 1 | 1 / 0 and 2 tie | Similar among batch jobs | Same-nice ID 2 behaves like RR 0. Shorter quanta should reduce wakeup waits, with added switching overhead. |
| 3: different nice values | 2 / 1 | 1 / 2 | Best 0, 1, and 6 roughly tied; worst 2 | Strict priority may finish high-priority batch work early but delay lower-priority jobs and reduce weighted fairness. |
| 4: staggered short jobs, equal nice | 0 and 2 tie / 1, tentative | 1 / 0 and 2 tie | N/A | ID 2 behaves like RR 0. RR 1 should dispatch arrivals sooner, but switching overhead may raise overall turnaround. Early batch jobs are expected to finish before all batch arrivals, preventing a fairness window. |
| 5: I/O-intensive nice-0 and batch nice-11 jobs | 0 / 2, tentative | 1 / 2 | Similar between the two batch jobs | ID 2 prioritizes I/O jobs and may defer batch work. Equal-nice batch jobs may still score fairly relative to each other even if both are delayed. |

These are hypotheses, not results. The turnaround ordering for workloads 4 and 5 is especially uncertain because response benefits compete with switch overhead. Fairness is a batch-only metric, not whole-workload equity.

## 1. Add Scheduler ID 6 to `policy.c`

Register a distinct policy ID so `set_scheduler(6)` is accepted. Keep scheduler IDs 0, 1, and 2 unchanged.

## 2. Implement Round Robin with a 5-Tick Quantum

Reuse the existing round-robin picker. ID 6 differs from ID 0 only by setting the quantum to 5 ticks.

## 3. Add the Scheduler Comment

The policy list at the top of `kernel/policy.c` identifies Feature 7 / scheduler ID 6 as round robin with a 5-tick quantum.

## 4. Review and Stage the Diff

Reviewed with `git diff -- kernel/policy.c`; the only policy changes are the ID 6 definition/comment and its quantum initialization. The policy diff was staged and committed separately, without the notebook or CSV artifacts.

## 5. Build xv6

`make -s kernel/kernel fs.img` succeeded. The linker emitted an RWX LOAD-segment warning.

## 6. Test Scheduler ID 6 in xv6 Shell

Before the sweep, booted QEMU using an isolated filesystem image and ran `wl 1 6`. It completed and returned to the shell. The interactive summary was turnaround 719.40 ticks, response 5.20 ticks, fairness 0.999, and switch overhead 3.67%.

## 7. Run the Workload Sweep

After the interactive test passed, ran all workloads with schedulers 0, 1, 2, and 6. The active user QEMU held the workspace `fs.img`, so the test used a fresh filesystem image under `/tmp` through a `QEMUOPTS` override. Results are in `scheduler_summary.csv` and `scheduler_jobs.csv`.

## 8. Tabulate Sweep Results

The summary table reports mean batch turnaround, mean response over all CPU bursts, nice-weighted batch fairness, switch overhead, and worst response. Workload 4 fairness is N/A because no fairness window opens.

| Workload | Scheduler | Turnaround (ticks) | Response (ticks) | Fairness | Overhead (%) | Worst response (ticks) |
|---:|---:|---:|---:|---:|---:|---:|
| 1 | 0 | 706.80 | 5.20 | 0.997 | 1.88 | 9 |
| 1 | 1 | 857.00 | 0.40 | 0.999 | 18.44 | 1 |
| 1 | 2 | 706.80 | 5.20 | 0.997 | 1.88 | 9 |
| 1 | 6 | 719.40 | 5.20 | 0.999 | 3.67 | 17 |
| 2 | 0 | 1074.00 | 17.42 | 0.999 | 3.01 | 24 |
| 2 | 1 | 1284.00 | 1.35 | 0.999 | 20.00 | 3 |
| 2 | 2 | 1074.00 | 17.42 | 0.999 | 3.01 | 24 |
| 2 | 6 | 1096.00 | 7.04 | 0.999 | 4.71 | 17 |
| 3 | 0 | 1277.50 | 27.29 | 0.393 | 2.39 | 30 |
| 3 | 1 | 1558.50 | 1.11 | 0.393 | 20.00 | 3 |
| 3 | 2 | 810.25 | 65.67 | 0.250 | 1.00 | 969 |
| 3 | 6 | 1307.50 | 8.76 | 0.393 | 4.29 | 15 |
| 4 | 0 | 216.50 | 12.66 | N/A | 2.04 | 27 |
| 4 | 1 | 267.58 | 1.08 | N/A | 18.89 | 3 |
| 4 | 2 | 216.50 | 12.66 | N/A | 2.04 | 27 |
| 4 | 6 | 220.41 | 7.50 | N/A | 3.90 | 14 |
| 5 | 0 | 425.00 | 7.04 | 0.999 | 8.37 | 25 |
| 5 | 1 | 1124.00 | 3.25 | 0.999 | 20.00 | 4 |
| 5 | 2 | 1490.00 | 8.94 | 0.999 | 8.37 | 1192 |
| 5 | 6 | 560.00 | 7.07 | 0.999 | 8.76 | 15 |

## 9. Hand-Check One Batch Job

From the per-job CSV row for workload 1, scheduler 6, job 4: arrival = 43, first_run = 60, completion = 313. Thus turnaround = completion - arrival = 313 - 43 = 270 ticks. Initial response = first_run - arrival = 60 - 43 = 17 ticks. The CSV row reports turnaround 270, `resp_count` 1, and `resp_sum` 17, so both calculations match. Since this job has one CPU burst, its mean response also equals its initial response.

## Review Note

Al's plan was not available in the workspace or conversation, so its specific fairness and clarity have not been reviewed.

# MLFQ Scheduler 3: Pre-Implementation Record

Prediction recorded 2026-11-10, before scheduler 3 implementation or MLFQ runs.

## Workload 2 Prediction

Workload 2 has three nice-0 batch jobs needing 300 CPU ticks each and three nice-0 I/O jobs with short CPU bursts and repeated sleeps. Assuming the project mapping places nice 0 in Q1 and the supplied table confirms Rule 4's 10-tick Q1 allotment, I predict MLFQ response will be between scheduler 1 (1-tick RR, lowest response) and scheduler 0 (10-tick RR, highest response). I tentatively predict batch turnaround between scheduler 0 (best) and scheduler 1 (worst), as queue boosts and shorter service slices may add switching overhead. Batch-only fairness should remain similarly high because the three batch jobs have equal nice. These are hypotheses; exact per-queue quanta/allotments remain to be confirmed from the course table, and may change these rankings.

The full course table was supplied on 2026-11-11 and confirmed the Q1/nice-0 assumptions used in this pre-run prediction. The prediction above is preserved as written.

| Queue | Nice range for starting level | Quantum (ticks) | Allotment (ticks) |
|---|---:|---:|---:|
| Q2 (highest) | nice <= -10 | 2 | 2 |
| Q1 | -10 < nice <= 10 | 10 | 10 |
| Q0 (lowest) | nice > 10 | 15 | no further demotion |

## Rule-Check Matrix: Expected Before Tests

Actual observation/evidence and pass/fail are intentionally blank until the tests are run. Workloads 2, 3, and 5 exercise Rules 1-5; Rule 6 requires the dedicated nice-change test.

| Test and setup | Expected observation | Actual observation and evidence | Pass or fail |
|---|---|---|---|
| Rule 1: workload 5, `logwl 5 3`; a nice-11 Q0 batch process is running when a nice-0 Q1 I/O process becomes runnable. | The Q1 process runs no later than the next timer tick after becoming runnable; Q0 does not continue through its lower-level quantum. | Pending | Pending |
| Rule 2: workload 2, `logwl 2 3`; observe equal-priority, same-level runnable batch jobs over several dispatches. | Their `running PID at tick` lines rotate in round-robin order within the queue; none is skipped indefinitely. | Pending | Pending |
| Rule 3: workload 3, `logwl 3 3`; correlate worker PIDs/nice values with start-level assignments. | Nice -10 starts Q2, nice -5 and 5 start Q1, and nice 11 starts Q0. | Pending | Pending |
| Rule 4: a nice-0 child runs four charged CPU ticks, blocks/sleeps, then resumes for six more charged CPU ticks under scheduler 3 with logging enabled. | It starts Q1; its `used` count carries across the block; after 10 total charged ticks it changes Q1->Q0. The transition tick is when the tenth CPU tick is charged, not simply 10 wall-clock ticks after start. | Pending | Pending |
| Rule 5: workload 5, `logwl 5 3`; observe the nice-11 batch workers across the 60-tick boost boundary. | A worker starting in Q0 is boosted to Q1 at the 60-tick boundary, and its allotment use resets; repeated boosts prevent indefinite Q1 starvation of Q0 batch jobs. | Pending | Pending |
| Rule 6: dedicated `mlfqtest`; change a child nice value first within the same starting-level band, then across a band. | Same-band change leaves queue and `used` unchanged; cross-band change moves to the new starting queue and resets `used` to zero. | Pending | Pending |

## Test Plan

The complete queue table and nice mapping are now confirmed. Implement scheduler 3 and logging-only queue-transition messages; review the diff and build; run `logwl 2 3`, `logwl 3 3`, and `logwl 5 3`, using worker queue-change and running lines as evidence. Add `user/mlfqtest.c` to `UPROGS` for Rule 6 and for a controlled sleep-persistent allotment check; test it under scheduler 3. Fill the matrix from actual logs, then run the unlogged scheduler sweep with ID 3 included. Logged workload metrics are not for the results table because logging changes timing.

# CFS-Like Scheduler ID 4: Pre-Run Prediction, Checks, and Results

Prediction recorded 2026-11-11 before implementing ID 4 or running it.

## Prediction vs. MLFQ

**Workload 2:** all six jobs have nice 0. I expect the batch-only, nice-weighted fairness score to be similarly high for CFS-like scheduling and MLFQ. I expect MLFQ to have better response for I/O-waking jobs because its priority queues favor recently short-running tasks, while CFS aims for equal weighted CPU share rather than interactive preference. CFS batch turnaround is uncertain; I tentatively expect it to be comparable to or slightly better than MLFQ because it does not demote batch processes below I/O jobs, but switch overhead and workload timing can reverse this.

**Workload 3:** I expect CFS-like scheduling to score better than MLFQ on the harness's nice-weighted batch fairness metric because it uses the full weight table instead of coarse priority levels and periodic boosts. I expect MLFQ to have better response for the high-priority nice -10 interactive job. CFS should give the nice-11 batch job a positive weighted share, potentially reducing its worst response/turnaround compared with MLFQ. Mean batch turnaround is uncertain because MLFQ may finish high-priority jobs earlier. These are hypotheses, not results.

The harness fairness metric is a nice-weighted Jain score over batch jobs only; it is not fairness across all batch and interactive jobs.

## Rule-Check Matrix

| Test and setup | Expected observation | Actual observation and evidence | Pass or fail |
|---|---|---|---|
| Weighted shares: `wl 3 4`; compare nice -10, -5, 5, and 11 batch jobs. | CPU service during the fairness window, divided by each job's nice weight, is approximately equal; nice-weighted fairness approaches 1. | `cfs_summary.csv`: workload 3, scheduler 4 fairness = 0.997. Per-job rows in `cfs_jobs.csv` show all four batch jobs received their 300-tick CPU demand and completed. | Pass |
| Low-weight progress: `wl 3 4`; inspect the nice-11 batch job's per-job row. | The job receives nonzero CPU and completes despite the higher-weight jobs; it is not starved. | `cfs_jobs.csv` row `3,4,3`: nice 11, batch, completion 1358, CPU 30000 units (300 ticks), response 3 ticks. | Pass |
| Wakeup response: `wl 2 4`; inspect the three I/O jobs' response counts and worst response. | Each I/O return becomes runnable and receives service without gaining a lead from sleep; all bursts complete with bounded response. | `cfs_jobs.csv` workload 2 scheduler 4: I/O jobs have response counts 30/20/20 and worst responses 2/3/2 ticks; all complete. The per-job output shows the outcome, not internal `vrt` values. | Pass, behavioral evidence |
| Long-wait progress: `wl 5 4`; inspect per-job CPU and completion for the nice-11 batch jobs. | Both low-weight batch jobs receive CPU and complete despite repeated I/O wakeups from nice-0 jobs. | `cfs_jobs.csv` rows `5,4,0` and `5,4,1`: both nice-11 batch jobs receive 15000 CPU units (150 ticks) and complete at ticks 1568 and 1569. | Pass |

## Unlogged Sweep Results

The complete sweep for workloads 1-5 and schedulers 0, 1, 2, 3, 4, and 6 is saved in `cfs_summary.csv` and `cfs_jobs.csv`. Logged timings are excluded. Selected ID 3 versus ID 4 comparisons:

| Workload | Scheduler | Turnaround (ticks) | Response (ticks) | Fairness | Overhead (%) | Worst response |
|---:|---:|---:|---:|---:|---:|---:|
| 2 | 3 (MLFQ) | 1059.33 | 9.32 | 0.996 | 3.28 | 34 |
| 2 | 4 (CFS) | 1284.00 | 0.98 | 0.999 | 20.00 | 3 |
| 3 | 3 (MLFQ) | 1182.25 | 9.44 | 0.424 | 2.51 | 93 |
| 3 | 4 (CFS) | 940.00 | 0.17 | 0.997 | 7.99 | 3 |
| 5 | 3 (MLFQ) | 869.50 | 7.03 | 0.996 | 8.37 | 45 |
| 5 | 4 (CFS) | 1559.50 | 2.16 | 0.999 | 20.00 | 3 |

The workload-2 response prediction was wrong: CFS had lower aggregate response than MLFQ, but higher batch turnaround. On workload 3, CFS substantially improved batch-only nice-weighted fairness and aggregate response, while also lowering mean batch turnaround. On workload 5, CFS improved response but had worse batch turnaround; its fairness score does not describe the I/O jobs' share. CFS's 20% overhead in workloads 2 and 5 reflects frequent switching and is an important tradeoff.

The sweep build printed `mkfs` output despite the attempt to preserve the workspace image via `MAKEFLAGS`; the active user QEMU may have had its attached `fs.img` rewritten. I did not stop or modify that QEMU session.

# Feature 4: MLFQ Tuning Plan, Predictions, and Results

Predictions recorded 2026-11-11 before implementation of IDs 10-12. Baseline is scheduler 3. Each variant changes one setting, retains its own ID, and ran on all public workloads 1-5.

## Baseline Diagnosis

Scheduler 3's lowest fairness is workload 3: 0.424. Its nice-11 batch job arrives at 0, first runs at 93, and completes at 1287; the nice -10 batch job arrives and first-runs at 0 and completes at 1022. Other batch first-run times are 12 and 15, with completions 1262 and 1158. This supports a long wait for the low-priority batch job. The fairness score itself depends on nice-weighted CPU service during the fairness window, not arrival/first-run/completion alone.

## Tuning Matrix

| Round | Hypothesis | Change (ID and what changed) | Prediction | Actual result | Explanation |
|---|---|---|---|---|---|
| 1 | The 60-tick boost interval contributes to the workload-3 nice-11 job's 93-tick wait and poor fairness. | ID 10: boost interval 30 ticks; other ID 3 settings unchanged. | W3 fairness improves over 0.424 and nice-11 first-run is earlier; more boosts may raise response/overhead elsewhere. | W3 fairness 0.468, turnaround 1055.00 vs ID 3's 0.424 and 1182.25; nice-11 first-run 44 vs 93. But W2 response worsened 9.32->12.02 and fairness 0.996->0.923. W5 turnaround improved 869.50->481.00. | Prediction held for the target fairness and W5 turnaround, but more frequent boosts also reset allotments/promote batch work, disrupting the workload-2 interactive advantage. W3 fairness improvement is modest, not a full fix. |
| 2 | A 10-tick Q1 quantum gives same-level processes longer waits between turns. | ID 11: Q1 quantum 5 ticks; Q1 allotment stays 10. | W2 response/worst response improve; batch turnaround may rise slightly from more switches. | W2 response 9.32->4.76, worst 34->18; turnaround 1059.33->1079.00 and overhead 3.28%->4.17%. W3 fairness fell 0.424->0.369. | Shorter slices improved interactive response but added switches and changed the relative service in W3; the fairness regression shows the improvement is not global. |
| 3 | A longer Q0 slice could reduce switch overhead but make Q0 peers wait longer. | ID 12: Q0 quantum 30 instead of 15; Q0 allotment and all other settings unchanged. | W5 batch turnaround may improve modestly; Q0 response may worsen. W3 fairness may shift. | W5 turnaround only improved 869.50->865.50; response 7.03->7.16, worst 45->59, overhead 8.37%->8.33%. The two nice-11 jobs first-ran at 40/70 and completed at 843/906, versus ID 3 first-runs 40/55 and completions 848/909. W3 fairness declined 0.424->0.401. | The larger Q0 slice barely changed total overhead/turnaround because higher queues frequently preempt Q0. It delayed the second Q0 batch job by 15 ticks and raised worst response, matching the predicted latency risk. |

## All-Workload Results

Values are `turnaround ticks / response ticks / fairness / overhead %`. Fairness is N/A for workload 4.

| Workload | ID 3 baseline | ID 10 | ID 11 | ID 12 |
|---:|---|---|---|---|
| 1 | 688.80 / 2.80 / 0.983 / 1.49 | 711.60 / 11.20 / 0.993 / 1.75 | 693.00 / 1.80 / 0.975 / 2.53 | 674.40 / 2.80 / 0.983 / 1.22 |
| 2 | 1059.33 / 9.32 / 0.996 / 3.28 | 982.33 / 12.02 / 0.923 / 3.05 | 1079.00 / 4.76 / 0.999 / 4.17 | 1054.66 / 9.32 / 0.997 / 3.14 |
| 3 | 1182.25 / 9.44 / 0.424 / 2.51 | 1055.00 / 2.79 / 0.468 / 3.39 | 1250.00 / 5.52 / 0.369 / 3.60 | 1227.25 / 9.44 / 0.401 / 2.30 |
| 4 | 196.25 / 2.25 / N/A / 1.71 | 216.00 / 7.75 / N/A / 1.91 | 193.91 / 1.08 / N/A / 2.47 | 188.50 / 2.00 / N/A / 1.34 |
| 5 | 869.50 / 7.03 / 0.996 / 8.37 | 481.00 / 7.12 / 0.999 / 8.37 | 922.00 / 7.19 / 0.998 / 8.74 | 865.50 / 7.16 / 0.993 / 8.33 |

The complete summary and per-job files are `mlfq_tuning_summary.csv` and `mlfq_tuning_jobs.csv`. Every variant was smoke-tested (`wl 1 10`, `wl 1 11`, `wl 1 12`) before the corrected sweep. An initial sweep revealed the new IDs were missing from MLFQ dispatch/tick/wakeup/nice routing; that was fixed, the kernel rebuilt, all three IDs smoke-tested again, and the full sweep rerun. Only the corrected sweep above is used here.

# Feature 5: Design for Scheduler ID 5

Design statement recorded 2026-11-11 before implementation.

## Design

ID 5 is CFS-like weighted virtual-runtime scheduling with a four-tick minimum slice. It targets ID 4's 20% switch overhead on workloads 2 and 5, and especially its workload-5 batch turnaround of 1559.50 ticks. It keeps charging weighted virtual runtime on every CPU tick and still selects the runnable process with the smallest virtual runtime, but does not preempt for a virtual-runtime comparison until the current process has received at least four ticks. The expected cost is worse response for waking I/O jobs and somewhat coarser short-term weighted fairness. The rule depends only on scheduler ID, elapsed quantum ticks, process state, nice weight, and virtual runtime; it does not recognize workloads.

This adapts the latency-versus-batching tradeoff described in the official [Linux CFS design documentation](https://docs.kernel.org/scheduler/sched-design-CFS.html), which documents `base_slice_ns` as a tunable between lower latency and improved batching. The 4-tick setting is an xv6 experiment hypothesis, not a Linux default.

## Prediction Before Implementation

Predictions are relative to measured scheduler ID 4; actual results will be entered after the final sweep.

| Workload | Predicted turnaround | Predicted response | Predicted fairness | Why |
|---|---|---|---|---|
| 1: equal-nice batch | Improve modestly from 857.00 ticks. | Worsen from 0.40 ticks. | Stay near 0.999. | Fewer context switches among equal-weight jobs, but each arrival may wait for a longer minimum slice. |
| 2: equal-nice batch and I/O | Improve from 1284.00 ticks as overhead falls. | Worsen from 0.98 ticks, especially for I/O wakeups. | Stay near 0.999 among batch jobs. | The minimum slice reduces switching but delays short bursts; batch jobs remain equal-weight. |
| 3: mixed nice values | Similar or slightly improve from 940.00 ticks if lower overhead outweighs batching delay. | Worsen from 0.17 ticks. | Decline modestly from 0.997. | Per-tick charging preserves weighted long-run shares, but a four-tick minimum can increase short-term service skew. |
| 4: staggered short jobs | Improve from 267.58 ticks through fewer switches. | Worsen for newly arrived short jobs. | N/A in the harness. | The longer minimum slice can delay short arrivals; fairness is not measured for this workload. |
| 5: I/O-intensive nice-0 and batch nice-11 | Improve substantially from 1559.50 ticks. | Worsen from 2.16 ticks for I/O bursts. | Remain high among the two batch jobs. | Fewer preemptions may reduce switch loss and help batch completion, while the fairness metric excludes interactive jobs. |

Implementation plan: add ID 5 without changing IDs 0-4, 6, or 10-12; reset the CFS minimum-slice setting and tie-breaking cursor in `policy_init()`; reuse existing `vrt`, `qticks`, and wakeup clamping; keep `vrt` charging on every tick; then build, smoke-test `wl 2 5`, and run the final all-workload comparison. No source changes have been made yet.

# Final Scheduler Comparison

Final unlogged sweep: workloads 1-5, schedulers 0, 1, 2, 3, 4, 10 (best tuned MLFQ), and 5 (custom). ID 6 is omitted as requested. Values are turnaround ticks / response ticks / fairness / switch overhead %. Workload 4 fairness is N/A. Full CSVs: `final_summary.csv` and `final_jobs.csv`.

| Workload | ID 0 | ID 1 | ID 2 | ID 3 | ID 4 | ID 10 | ID 5 |
|---:|---|---|---|---|---|---|---|
| 1 | 706.80 / 5.20 / 0.997 / 1.88 | 857.00 / 0.40 / 0.999 / 18.44 | 706.80 / 5.20 / 0.997 / 1.88 | 688.80 / 2.80 / 0.983 / 1.49 | 857.00 / 0.40 / 0.999 / 18.44 | 711.60 / 11.20 / 0.993 / 1.75 | 724.80 / 2.00 / 0.999 / 4.59 |
| 2 | 1074.00 / 17.42 / 0.999 / 3.01 | 1284.00 / 1.35 / 0.999 / 20.00 | 1074.00 / 17.42 / 0.999 / 3.01 | 1059.33 / 9.32 / 0.996 / 3.28 | 1284.00 / 0.98 / 0.999 / 20.00 | 982.33 / 12.02 / 0.923 / 3.05 | 1104.00 / 6.63 / 0.999 / 5.54 |
| 3 | 1277.50 / 27.29 / 0.393 / 2.39 | 1558.50 / 1.11 / 0.393 / 20.00 | 810.25 / 65.67 / 0.250 / 1.00 | 1182.25 / 9.44 / 0.424 / 2.51 | 940.00 / 0.17 / 0.997 / 7.99 | 1055.00 / 2.79 / 0.468 / 3.39 | 891.50 / 1.76 / 0.980 / 2.51 |
| 4 | 216.50 / 12.66 / N/A / 2.04 | 267.58 / 1.08 / N/A / 18.89 | 216.50 / 12.66 / N/A / 2.04 | 196.25 / 2.25 / N/A / 1.71 | 267.58 / 1.08 / N/A / 18.89 | 216.00 / 7.75 / N/A / 1.91 | 219.41 / 4.33 / N/A / 4.77 |
| 5 | 425.00 / 7.04 / 0.999 / 8.37 | 1124.00 / 3.25 / 0.999 / 20.00 | 1490.00 / 8.94 / 0.999 / 8.37 | 869.50 / 7.03 / 0.996 / 8.37 | 1559.50 / 2.16 / 0.999 / 20.00 | 481.00 / 7.12 / 0.999 / 8.37 | 1499.00 / 5.34 / 0.999 / 8.96 |

ID 5 reduced CFS overhead and response latency relative to ID 4 on most mixed workloads, and achieved 0.980 fairness on workload 3. However, the prediction that it would substantially improve workload-5 batch turnaround was wrong: turnaround was 1499 ticks versus ID 4's 1559.50, still far behind MLFQ ID 10's 481 ticks and RR ID 0's 425 ticks. Four-tick slices cannot overcome the nice-weight share difference for the two nice-11 batch jobs when nice-0 I/O jobs repeatedly return; the throughput/turnaround target needs a policy-level group/share or stronger low-weight protection rather than only a minimum slice.